# G1M1: Baseline vs Huber vs Ridge vs Huber+Ridge (Unified Comparator)

This notebook runs 4 linear-term models on the same preprocessed data and compares them with `UnifiedModelComparator`.

- Baseline: `Urc1` (OLS)
- Huber: `Urc1BaseHuber(epsilon=1.00, alpha=0.0)`
- Ridge: `Urc1BaseHuber(epsilon=1e6, alpha=0.1)`
- Huber+Ridge: `Urc1BaseHuber(epsilon=1.00, alpha=0.1)`

Comparison enables:
- Cond metrics: `include_all_cond_metrics=True`
- GT metrics: `include_gt_metrics=True` (if GT bundle is found)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_huber import Urc1BaseHuber
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [ ]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G1M1_new.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\..\\plots\\huber\\G1M1_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.3,
        "Tref": 58,
        "OHref": 11,
        "gt_file": r"..\\..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 100,
        "gt_file": r"..\\..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.48,
        "Tref": 57,
        "OHref": 100,
        "gt_file": r"..\\..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",
    },
}


# Model fitting common config (shared across all models)
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
    "data_filter_h_since_last_start_min": 0.5,
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [ ]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)
preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name
print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

In [ ]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}
print("\n  -> Training Baseline (OLS) model...")
urc_baseline = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    nonlinear_term="I2",
    **COMMON_CONFIG,
)
models["Baseline (I2)"] = urc_baseline
print("  ✓ Baseline training completed")

print("\n  -> Training Huber model...")
urc_huber = Urc1BaseHuber(
    data=data,
    name=dataset_name,
    epsilon=1.00,
    alpha=0.0,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Huber"] = urc_huber
print("  ✓ Huber training completed")

print("\n  -> Training Ridge model...")
urc_ridge = Urc1BaseHuber(
    data=data,
    name=dataset_name,
    epsilon=1e6,
    alpha=0.1,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Ridge"] = urc_ridge
print("  ✓ Ridge training completed")

print("\n  -> Training Huber+Ridge model...")
urc_hr = Urc1BaseHuber(
    data=data,
    name=dataset_name,
    epsilon=1.35,
    alpha=0.1,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Huber+Ridge"] = urc_hr
print("  ✓ Huber+Ridge training completed")

print(f"\n✓ {len(models)} models trained successfully\n")

In [ ]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"✓ UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()
    print(f"Looking for GT file: {gt_path}")

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column in {gt_path}. Columns: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  ✓ Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  ✗ Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  ⚠ GT file NOT found: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\n{'=' * 80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics will be {'ENABLED ✓' if has_gt else 'DISABLED'}")
print(f"{'=' * 80}\n")

In [ ]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n▓▓▓ REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref}) ▓▓▓")

    df_metrics = comparator.compare_all(
        i_target=iref,
        outlier_threshold_method="2rmse",
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )
    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
            uncertainty_style="band",
            uncertainty_opacity=0.12,
            show_series_line=True,
            rate_precision=6,
        )
        print("✓ Trend plot finished")
    except Exception as e:
        print(f"⚠ Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

In [ ]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R² distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("✓ Fit quality completed")
except Exception as e:
    print(f"⚠ Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=False)
    print("✓ Coefficient diagnostic completed")
except Exception as e:
    print(f"⚠ Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("✓ Coverage Gantt completed")
except Exception as e:
    print(f"⚠ Coverage Gantt failed: {e}")

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\n✓ Trained models: {len(models)}")
print(f"✓ Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"✓ Ground truth data: {'Loaded ✓' if has_gt else 'Not available'}")
print(f"✓ Output plots: {PLOTS_OUTPUT_DIR}/" if SAVE_PLOTS else "✓ Plots displayed (not saved)")
print("\nKey settings:")
print(f"  - All condition metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - GT metrics: {SHOW_GT_METRICS}")
print(f"  - Save plots: {SAVE_PLOTS}")